# TaskGenerator Quickstart

This notebook shows how to use `TaskGenerator` with:
- `task_callable` (dynamic tasks)
- `tasks=[(env, info), ...]` (static tasks)

In [ ]:
from pathlib import Path
import sys

# Make repository root importable when running from examples/
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'examples' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)

## Dynamic env creation

If you want to use a function to generate the envs, define the function as below.
The function should have at least the `random_seed` parameter.

In [2]:
from typing import Dict, Any, Tuple
import gymnasium as gym

from src.task_generator import TaskGenerator

# Have to integrate thr random_seed parameter !
def make_cartpole_task(random_seed: int, env_id: str = 'CartPole-v1') -> Tuple[gym.Env, Dict[str, Any]]:
    env = gym.make(env_id)
    env.reset(seed=random_seed)
    info = {'env_id': env_id, 'seed': random_seed}
    return env, info

In [3]:
# Dynamic task generation with revisits
tg_dynamic = TaskGenerator(
    task_callable=make_cartpole_task,
    task_callable_params={'env_id': 'CartPole-v1'},
    revisit_ratio=0.5,
    revisit_start=2,
    sampling_method='random',
    generator_seed=123,
)

for meta_step in range(6):
    env, task_info, origin_meta_step = tg_dynamic.get_task(meta_step)
    print(f'meta_step={meta_step} | origin={origin_meta_step} | info={task_info}')
    env.close()

meta_step=0 | origin=0 | info={'env_id': 'CartPole-v1', 'seed': 6293585094778680549}
meta_step=1 | origin=1 | info={'env_id': 'CartPole-v1', 'seed': 496411279815456401}
meta_step=2 | origin=0 | info={'env_id': 'CartPole-v1', 'seed': 6293585094778680549}
meta_step=3 | origin=1 | info={'env_id': 'CartPole-v1', 'seed': 496411279815456401}
meta_step=4 | origin=0 | info={'env_id': 'CartPole-v1', 'seed': 6293585094778680549}
meta_step=5 | origin=5 | info={'env_id': 'CartPole-v1', 'seed': 8516354435172480967}


## Static env creation

In [4]:
# Static list of tasks
static_tasks = [
    (gym.make('CartPole-v1'), {'task_name': 'cp_0'}),
    (gym.make('CartPole-v1'), {'task_name': 'cp_1'}),
]

tg_static = TaskGenerator(
    tasks=static_tasks,
    sampling_method='cyclic',
)

for meta_step in range(4):
    env, task_info, origin_meta_step = tg_static.get_task(meta_step)
    print(f'meta_step={meta_step} | origin={origin_meta_step} | info={task_info}')

for env, _ in static_tasks:
    env.close()

meta_step=0 | origin=None | info={'task_name': 'cp_0'}
meta_step=1 | origin=None | info={'task_name': 'cp_1'}
meta_step=2 | origin=None | info={'task_name': 'cp_0'}
meta_step=3 | origin=None | info={'task_name': 'cp_1'}


## Custom Interface 

The `TaskGenerator` class is provided to facilitate the creation, sampling and revisit of task especially for complex environments that need specific initialization logic. One can create its own task generator as long as it implements the 2 necessary methods called during the meta-learning process:
- get_tasks: that generates or retreive a env
- reset_history: that resets the list of already sampled tasks:

In [5]:
from typing import Optional

In [6]:
class SimpleCartPoleTaskGenerator:
    """
    Minimal task-generator API compatible with BaseMetaAlgorithm.
    """

    def __init__(self, env_id: str = 'CartPole-v1'):
        self.env_id = env_id

    def reset_history(self) -> None:
        pass

    def get_task(
        self,
        meta_step: int,
        seed: Optional[int] = None,
    ) -> Tuple[gym.Env, Dict[str, Any], Optional[int]]:
    
        env = gym.make(self.env_id)
        env.reset(seed=seed if seed is not None else meta_step)
        info: Dict[str, Any] = {'meta_step': meta_step, 'seed': seed}
        return env, info, meta_step

In [7]:
tg_custom = SimpleCartPoleTaskGenerator(env_id="CartPole-v1")

In [8]:
env, info, meta_step = tg_custom.get_task(0, seed=123)
env.reset()

(array([-0.03240941,  0.03120945,  0.0423345 , -0.02234256], dtype=float32),
 {})